# 500-image Faster R-CNN baseline — Colab T4

Infrastructure only: this notebook has not been executed and contains no training results. Select **Runtime → Change runtime type → T4 GPU**. Obtain the dataset separately through its authorized source; never upload data or executed medical-image outputs to Git.

The unchanged subset script selects exactly 500 abnormal images. We retain every valid reader box without fusion. A seeded image-level 80/20 split prevents image overlap, but does not establish patient-level independence. This abnormal-only validation set cannot measure normal-image false positives. COCO-style mAP here scores reader annotations, not consensus lesions.

Pipeline: pydicom modality rescaling → padding-aware per-image min/max → MONOCHROME1 inversion → three grayscale channels → square 512-pixel resize with independent x/y box scales → COCO-pretrained Faster R-CNN ResNet-50 FPN with 14 finding classes plus background. Torchvision applies its own ImageNet mean/std normalization. No VOI windowing, augmentation, or reader-box fusion is applied.

API references: [torchvision Faster R-CNN](https://docs.pytorch.org/vision/0.26/_modules/torchvision/models/detection/faster_rcnn.html), [pydicom pixel processing](https://pydicom.github.io/pydicom/dev/reference/pixels.processing.html).


## 1. Clone or use the repository

These new files must be available in your Colab checkout. If they are not yet on GitHub, upload them to the matching paths in the cloned repository. No commit or push is performed by this notebook. Optionally set REF to a branch/tag/commit containing the infrastructure.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/Tunahantopal/vindr-cxr-medical-ai.git"
REPO = Path("/content/vindr-cxr-medical-ai")
REF = None
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
if REF:
    subprocess.run(["git", "-C", str(REPO), "checkout", REF], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
required = ["src/data/dicom_reader.py", "src/data/dataset.py",
            "src/data/preprocessing.py", "src/training/train.py", "configs/baseline.yaml"]
assert all(Path(p).is_file() for p in required), "Upload the new infrastructure files or select a revision containing them."
print("Repository revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Dependencies and CUDA

Keep Colab's paired CUDA-enabled torch/torchvision installation. Install only the additional training dependencies below; the EDA requirements are separate. If the torch/torchvision imports fail, start a fresh T4 runtime with a compatible pair. DICOM decoder plugins cover common compressed pixel formats.


In [ ]:
%pip install -q "pydicom>=3,<4" "PyYAML>=6,<7" "torchmetrics>=1.6,<2" "pycocotools>=2.0.8,<3" "pylibjpeg>=2,<3" "pylibjpeg-libjpeg>=2,<3" "pylibjpeg-openjpeg>=2,<3" matplotlib


In [ ]:
import torch
import torchvision
import yaml
from src.training.train import build_loaders, build_model, seed_everything, run

print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime before training in Colab."
print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda")


## 3. Access the separately obtained data

This example mounts your private Google Drive. Set DATA_ROOT to the directory containing the authorized full training CSV and a train/ folder of <image_id>.dicom files. The full annotation CSV is needed to reproduce the selection; at least the selected 500 DICOMs must be accessible. Links below point into ignored data/raw/ and do not redistribute anything. For faster I/O, you may instead stage the data privately under /content and update DATA_ROOT.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = Path("/content/drive/MyDrive/VinDr-CXR")  # Edit this path.
assert (DATA_ROOT / "train.csv").is_file()
assert (DATA_ROOT / "train").is_dir()
raw = REPO / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
for name in ("train.csv", "train"):
    target = raw / name
    source = DATA_ROOT / name
    if not target.exists() and not target.is_symlink():
        target.symlink_to(source, target_is_directory=source.is_dir())
    assert target.resolve() == source.resolve(), f"Existing {target} points elsewhere; resolve manually."
subprocess.run(["python", "-m", "src.data.create_subset"], check=True)


## 4. Lightweight synthetic checks and dataset setup

These checks use generated pixels only. They verify photometric polarity, constant-image handling, box scaling, and split invariance. The DataLoaders then verify subset/manifest agreement and DICOM availability. Missing or out-of-bounds boxes fail explicitly.


In [ ]:
import tempfile
import numpy as np
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, SecondaryCaptureImageStorage, generate_uid
from src.data.dicom_reader import read_dicom
from src.data.preprocessing import preprocess
from src.data.dataset import split_image_ids

with tempfile.TemporaryDirectory() as tmp:
    meta = FileMetaDataset()
    meta.TransferSyntaxUID = ExplicitVRLittleEndian
    meta.MediaStorageSOPClassUID = SecondaryCaptureImageStorage
    meta.MediaStorageSOPInstanceUID = generate_uid()
    path = Path(tmp) / "synthetic.dicom"
    ds = FileDataset(str(path), {}, file_meta=meta, preamble=bytes(128))
    ds.SOPClassUID = meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = meta.MediaStorageSOPInstanceUID
    ds.Rows, ds.Columns = 2, 4
    ds.SamplesPerPixel = 1
    ds.BitsAllocated = ds.BitsStored = 16
    ds.HighBit, ds.PixelRepresentation = 15, 0
    ds.RescaleSlope, ds.RescaleIntercept = 2, -10
    ds.PixelData = np.arange(8, dtype=np.uint16).reshape(2, 4).tobytes()
    decoded = []
    for mode in ("MONOCHROME2", "MONOCHROME1"):
        ds.PhotometricInterpretation = mode
        ds.save_as(path, enforce_file_format=True)
        decoded.append(read_dicom(path))
    np.testing.assert_allclose(decoded[0] + decoded[1], 1)
    np.testing.assert_allclose(decoded[0], np.arange(8).reshape(2, 4) / 7, atol=1e-6)
    ds.PixelPaddingValue = 0
    ds.save_as(path, enforce_file_format=True)
    assert read_dicom(path)[0, 0] == 0
    del ds.PixelPaddingValue
    ds.PixelData = np.ones((2, 4), dtype=np.uint16).tobytes()
    ds.save_as(path, enforce_file_format=True)
    assert np.isfinite(read_dicom(path)).all() and not read_dicom(path).any()
image, boxes = preprocess(decoded[0], [[1, 0, 4, 2]], 8)
assert image.shape == (3, 8, 8) and image.dtype == torch.float32
torch.testing.assert_close(boxes, torch.tensor([[2., 0., 8., 8.]]))
try:
    preprocess(decoded[0], [[0, 0, 5, 2]], 8)
except ValueError:
    pass
else:
    raise AssertionError("Out-of-bounds box was accepted")
ids = [f"synthetic-{i}" for i in range(500)]
a, b = split_image_ids(ids)
assert (a, b) == split_image_ids(list(reversed(ids)) + ids)
assert len(a) == 400 and len(b) == 100 and not set(a) & set(b)
assert set(a) | set(b) == set(ids)
print("Synthetic checks passed.")

config = yaml.safe_load(Path("configs/baseline.yaml").read_text())
seed_everything(config["seed"])
train_loader, val_loader, splits, class_names = build_loaders(config)
print("Split sizes:", {k: len(v) for k, v in splits.items()})
print("Classes absent from validation:", sorted(set(class_names) - {
    label for image_id in splits["validation"]
    for _, label in val_loader.dataset.annotations[image_id]}))


## 5. Inspect a batch before training

Confirm image polarity and box alignment. Titles omit source identifiers. Displayed medical images remain private runtime outputs; clear all outputs before sharing or putting a notebook in Git.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def show_detection(ax, image, target, threshold=None):
    ax.imshow(image.permute(1, 2, 0).cpu().numpy(), cmap="gray")
    for index, (box, label) in enumerate(zip(target["boxes"], target["labels"])):
        score = None if "scores" not in target else float(target["scores"][index])
        if threshold is not None and score is not None and score < threshold:
            continue
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=1))
        text = class_names[int(label)] + (f" {score:.2f}" if score is not None else "")
        ax.text(x1, y1, text, color="yellow", fontsize=6)
    ax.axis("off")

images, targets = next(iter(train_loader))
fig, axes = plt.subplots(1, len(images), figsize=(7 * len(images), 7), squeeze=False)
for ax, image, target in zip(axes.flat, images, targets):
    assert image.shape == (3, config["image_size"], config["image_size"])
    assert target["boxes"].shape[0] == target["labels"].shape[0] > 0
    assert target["labels"].dtype == torch.int64
    show_detection(ax, image, target)
plt.show()


## 6. Train and save measured metrics

The default is 10 epochs, batch size 2, SGD, full precision, 512×512 pixels. Reduce batch_size to 1 if T4 memory is insufficient. CPU execution is supported by the module, but Colab intentionally checks for CUDA above. Initial model construction downloads pretrained weights. Splits are deterministic; identical GPU results across software/hardware versions are not guaranteed.

Every epoch records mean train loss and its components, validation mAP@[.50:.95], AP50, AP75, and recall. The highest validation mAP selects best.pt; latest.pt is saved each epoch. No score threshold is applied before metric evaluation beyond the detector's default inference settings. This is a baseline on unfused reader boxes. The run saves config, split IDs, version information, optimizer/model state, and metrics under an ignored path. It refuses to overwrite a nonempty run directory. There is no automatic resume.


In [ ]:
from datetime import datetime, timezone
config["output_dir"] = "checkpoints/baseline_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
history = run(config)  # Executes training on the selected runtime.


In [ ]:
import json
output = Path(config["output_dir"])
metrics = json.loads((output / "metrics.json").read_text())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot([m["epoch"] for m in metrics], [m["train_loss"] for m in metrics])
axes[0].set(xlabel="Epoch", ylabel="Mean train loss")
axes[1].plot([m["epoch"] for m in metrics], [m["val_map"] for m in metrics])
axes[1].set(xlabel="Epoch", ylabel="Validation mAP")
plt.show()
print("Saved artifacts:", sorted(p.name for p in output.iterdir()))
# Record the runtime's resolved packages for later reproduction.
(output / "environment.txt").write_text(subprocess.check_output(["python", "-m", "pip", "freeze"], text=True))


## 7. Show validation predictions from the best checkpoint

The display threshold below is for visualization only. Empty predictions above the threshold are possible and should not be interpreted as normal findings.


In [ ]:
# Load only this run's locally generated checkpoint.
checkpoint = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
model = build_model(config["image_size"], pretrained=False)
model.load_state_dict(checkpoint["model"])
model.to(device).eval()
class_names = checkpoint["class_names"]
images = [val_loader.dataset[i][0] for i in range(min(3, len(val_loader.dataset)))]
with torch.inference_mode():
    predictions = model([image.to(device) for image in images])
fig, axes = plt.subplots(1, len(images), figsize=(7 * len(images), 7), squeeze=False)
for ax, image, prediction in zip(axes.flat, images, predictions):
    prediction = {k: v.cpu() for k, v in prediction.items()}
    show_detection(ax, image, prediction, threshold=0.3)
plt.show()


## 8. Preserve private run artifacts

Colab local storage is temporary. Copy checkpoints and metrics to private Drive storage before disconnecting. These artifacts include source image IDs in the split manifest/checkpoints. Do not place them or rendered images in a public repository. Clear notebook outputs before saving a shareable source notebook.


In [ ]:
import shutil
archive = Path("/content/drive/MyDrive/vindr_baseline_runs") / output.name
shutil.copytree(output, archive)  # Refuses to overwrite an existing archive.
print("Run copied to private Drive storage.")
